Notebook Python para ser executado no Google Colab onde iremos gerar os insights do Olist Store 

In [ ]:
# Import da bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração para melhorar a estética dos gráficos
sns.set_theme(style="whitegrid")
import warnings
warnings.filterwarnings('ignore')

Para carregar os dataset que estão no google drive, iremos nos conectar a ele.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Carregando os Datasets do drive, onde iremos considerar o caminho a partir da raiz o drive

In [ ]:
# Carregando as tabelas principais demonstradas no diagrama
customers = pd.read_csv('/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/olist_customers_dataset.csv')
geolocation = pd.read_csv('/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/olist_geolocation_dataset.csv')
order_items = pd.read_csv('/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/olist_order_items_dataset.csv')
payments = pd.read_csv('/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/olist_order_payments_dataset.csv')
reviews = pd.read_csv('/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/olist_order_reviews_dataset.csv')
orders = pd.read_csv('/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/olist_orders_dataset.csv')
products = pd.read_csv('/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/olist_products_dataset.csv')
sellers = pd.read_csv('/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/olist_sellers_dataset.csv')
product_category = pd.read_csv('/content/drive/MyDrive/1AIAT/Tech_Challange/FASE1/archive/product_category_name_translation.csv')

print("Datasets carregados com sucesso!")

Unindo Pedidos aos Clientes (customer_id)

In [ ]:
# Relaciona a tabela de pedidos com a tabela de dados dos clientes
df_orders_customers = pd.merge(orders, customers, on='customer_id', how='inner')

Unindo Itens do Pedido aos Produtos (product_id) e Vendedores (seller_id)

In [ ]:
# Relaciona itens do pedido com as características dos produtos
df_items_products = pd.merge(order_items, products, on='product_id', how='inner')

# Se quiser incluir os dados do vendedor também:
df_items_products_sellers = pd.merge(df_items_products, sellers, on='seller_id', how='inner')

Criando uma Tabela Mestre Analítica (Master Table)
Para gerar análises globais, podemos juntar a base de pedidos/clientes com a base de itens/produtos utilizando a chave order_id.

In [ ]:
# Unindo tudo em um único dataframe central para facilitar cruzamentos complexos
df_master = pd.merge(df_orders_customers, df_items_products, on='order_id', how='inner')

# Visualizando as primeiras linhas da tabela unificada
df_master.head()

Insight 1: Top 10 Categorias de Produtos que mais geram Receita

In [ ]:
# Agrupando por categoria de produto e somando o valor dos itens (price)
top_categories = df_master.groupby('product_category_name')['price'].sum().reset_index()
top_categories = top_categories.sort_values(by='price', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_categories, x='price', y='product_category_name', palette='viridis')
plt.title('Top 10 Categorias por Receita Total')
plt.xlabel('Receita Total (R$)')
plt.ylabel('Categoria do Produto')
plt.show()

Insight 2: Distribuição de Pedidos por Estado do Cliente (Geolocalização)

In [ ]:
# Contando quantos pedidos foram feitos por cada estado (customer_state)
orders_by_state = df_orders_customers['customer_state'].value_counts().reset_index()
orders_by_state.columns = ['state', 'order_count']

plt.figure(figsize=(12, 6))
sns.barplot(data=orders_by_state, x='state', y='order_count', palette='magma')
plt.title('Volume de Pedidos por Estado')
plt.xlabel('Estado')
plt.ylabel('Quantidade de Pedidos')
plt.show()

Insight 3: Relação entre o tipo de Pagamento e o Valor Gasto

In [ ]:
# Unindo pagamentos com os pedidos
df_payments_orders = pd.merge(payments, orders, on='order_id', how='inner')

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_payments_orders, x='payment_type', y='payment_value', showfliers=False)
plt.title('Distribuição do Valor dos Pagamentos por Tipo')
plt.xlabel('Tipo de Pagamento')
plt.ylabel('Valor do Pagamento (R$)')
plt.show()